## 한국어 뉴스 주제 분류 파인튜닝

기본 모델: `klue/bert-base`  
데이터셋: `klue/ynat`  
결과: 한국어 뉴스 제목을 주제별로 분류

파인튜닝 전에는 기본 모델이 뉴스 주제 분류용으로 학습되어 있지 않기 때문에 원하는 라벨을 제대로 예측하지 못한다.  
파인튜닝 후에는 한국어 뉴스 제목을 입력했을 때 정치, 경제, 사회, 생활문화, 세계, IT과학, 스포츠 중 하나로 분류할 수 있다.

영화리뷰 감정분석
기본 모델: beomi/kcbert-base
데이터셋: NSMC
분류 라벨: 부정, 긍정
결과: 한국어 영화 리뷰의 감정을 분류

In [1]:
!pip install -q -U "datasets<4.0.0" transformers evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate

print("GPU 사용 가능 여부:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

GPU 사용 가능 여부: True
GPU 이름: Tesla T4


데이터 불러오기

In [17]:
from datasets import load_dataset

dataset = load_dataset("klue/klue", "ynat")

model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["title"], padding="max_length", truncation=True, max_length=64)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["guid", "title", "url", "date"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

tokenized_datasets["train"] = tokenized_datasets["train"].select(range(5000))
tokenized_datasets["validation"] = tokenized_datasets["validation"].select(range(1000))

clf_metrics = evaluate.combine(["accuracy", "f1"])

metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)

    acc_result = metric_acc.compute(predictions=preds, references=labels)
    f1_result = metric_f1.compute(predictions=preds, references=labels, average="macro")

    return {
        "accuracy": acc_result["accuracy"],
        "f1_macro": f1_result["f1"]
    }

기본 모델과 토크나이저 불러오기

In [18]:
num_labels = 7

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    fp16=torch.cuda.is_available()
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

print("\n 파인튜닝(학습)을 시작합니다...")
trainer.train()

print("\n 검증 데이터셋 최종 평가 결과:")
eval_results = trainer.evaluate()
print(eval_results)

print("\n 실제 뉴스 제목으로 테스트를 진행합니다.")
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

label_map = {0: "IT과학", 1: "경제", 2: "사회", 3: "생활문화", 4: "세계", 5: "스포츠", 6: "정치"}

test_news_list = [
    "정부, 내년부터 반도체 산업에 수조 원 규모 보조금 지급 검토",
    "손흥민, 극적인 결승골로 팀의 대역전승 견인",
    "우주망원경이 포착한 새로운 은하계의 신비로운 모습",
    "여야, 민생 법안 처리를 위한 전격 합의안 도출",
    "출근길 중부지방 중심 기습 폭우, 지하철 운행 일부 지연"
]

for test_news in test_news_list:
    pred = classifier(test_news)[0]
    pred_label_id = int(pred['label'].split('_')[-1])

    confidence_percent = pred['score'] * 100

    print(f"입력 뉴스 제목: '{test_news}'")
    print(f"AI의 예측 결과: {label_map[pred_label_id]} (확신도: {confidence_percent:.1f}%)")
    print("-" * 50)


 파인튜닝(학습)을 시작합니다...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.017144,1.001565,0.856704,0.854711
2,0.049711,0.984146,0.855276,0.853834
3,0.081553,1.024460,0.858241,0.855019


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


 검증 데이터셋 최종 평가 결과:


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.081553,1.024460,3,0.858241,0.855019


{'eval_loss': 1.0244604349136353, 'eval_accuracy': 0.8582409135829582, 'eval_f1_macro': 0.8550192066620754}

 실제 뉴스 제목으로 테스트를 진행합니다.
입력 뉴스 제목: '정부, 내년부터 반도체 산업에 수조 원 규모 보조금 지급 검토'
AI의 예측 결과: 경제 (확신도: 41.7%)
--------------------------------------------------
입력 뉴스 제목: '손흥민, 극적인 결승골로 팀의 대역전승 견인'
AI의 예측 결과: 스포츠 (확신도: 100.0%)
--------------------------------------------------
입력 뉴스 제목: '우주망원경이 포착한 새로운 은하계의 신비로운 모습'
AI의 예측 결과: IT과학 (확신도: 100.0%)
--------------------------------------------------
입력 뉴스 제목: '여야, 민생 법안 처리를 위한 전격 합의안 도출'
AI의 예측 결과: 정치 (확신도: 100.0%)
--------------------------------------------------
입력 뉴스 제목: '출근길 중부지방 중심 기습 폭우, 지하철 운행 일부 지연'
AI의 예측 결과: 생활문화 (확신도: 100.0%)
--------------------------------------------------
